# Tutorial 1 - SPAC-seq held-out perturbation benchmark

This notebook reproduces the SPAC-seq benchmark numbers from a **trained CONCERT checkpoint**.
Nothing is retrained here: every cell below loads `model.pt` and runs inference.

### The question being scored

Five tissue sections (`1-1` ... `1-5`) carry a pooled CRISPR perturbation panel. A set of
perturbations is held out. For each held-out cell the model is given

* **where the cell sits** in the section (its array coordinates), and
* **what it looked like unperturbed** (its measured expression),

and is asked to predict the perturbed expression. The prediction is compared with the cells
that actually carry that perturbation, through the **log fold change over matched unperturbed
cells**.

### The two settings

| Code name | Manuscript name | What is held out |
|---|---|---|
| `seen` | **within-niche prediction** | the perturbation is removed from a patch of one slide but is still visible elsewhere - does its effect transfer to a new tissue context? |
| `unseen` | **cross-slide prediction** | the perturbation is removed from one target slide (85 knockout-target-slide pairs) and has to be carried over from the donor slides |

The code paths, output directories and CSV columns still say `seen` / `unseen`. Those names are
kept here because they are what is on disk; the manuscript uses the two names in the middle column.

### The scoring rule (benchmark v4), identical for every method

* **Prediction output.** Each source spot's predicted expression profile is turned into counts by
  one Poisson draw at that spot's measured library size (`--cf_output poisson_depth`). Every
  competitor and every reference strategy is scored under the same draw.
* **LFC reference.** The predicted log fold change is taken against the **measured** source spots
  (`--cf_seen_ref observed`); the measured log fold change is taken against disjoint unperturbed
  cells.
* **Genes.** A gene is scored when it is a highly variable gene of the target slide, is in the
  2,000-gene CONCERT panel, and is not the knocked-out gene (`genefix_wrap.py`).

### The two metrics

* **LFC Spearman** - do the right genes move, in the right direction? (higher is better)
* **LFC RMSE** (`lfc_dist` in the tables) - is the size of the response right? (lower is better)

E-distance appears in the JSON files. It is **retired**: it is not a reported metric and should
not be quoted.

### What the notebook does

1. resolve every path and check it exists
2. load the trained (clonality-gated) checkpoint into a live model
3. read the learned per-perturbation spatial footprint lengthscales out of the weights
4. make **one** counterfactual prediction by hand and look at it
5. score **one slide** with the exact function the SLURM array job calls, and check it against the
   shipped result
6. aggregate the five slides into the reported within-niche number
7. repeat for the cross-slide setting
8. put CONCERT next to the published methods and the reference strategies

## 0. Before you start

**Run this on a GPU node.** Steps 4-7 do real inference; on CPU they are unusably slow.

```bash
# interactive GPU session, then start Jupyter inside it
salloc -p gpu --gres=gpu:1 --mem=200G -c 8 -t 0-08:00
source /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/miniconda3_new/etc/profile.d/conda.sh
conda activate /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/concert_env
jupyter lab --no-browser --ip=0.0.0.0
```

The environment is `concert_env`. Note that it is pinned to CUDA 11.8 / torch 2.3 and **cannot
use Blackwell GPUs** - ask for an H100 or A100 (`--gres=gpu:nvidia_h200:1` on the Kempner
partitions) if the job dies with an `nvrtc -arch` error a few minutes in.

**Rough runtimes on one H100**

| Step | Time |
|---|---|
| 2 - load checkpoint + slide | 2-4 min |
| 4 - one prediction | < 1 min |
| 5 - score one slide (full panel) | 15-40 min |
| 7 - one cross-slide unit | 3-5 min |
| 6, 8 - reading shipped results | seconds |

Steps 5 and 7 are optional: set `RUN_SLIDE_EVAL = False` / `RUN_ONE_UNSEEN_UNIT = False` to skip
them and read the shipped results instead.

## 1. Paths

Everything the notebook touches is named here. The roots involved:

* **`CODEBASE`** - the released code (model source + scoring code).
* **`OUTPUTS`** - where training wrote the checkpoints.
* **`SCORING_LIB`** - the evaluation working tree, which holds the prepared cell sets, every
  method's per-record results, and the aggregated tables the paper reports
  (`v4_aggregate_sampled/`).
* **`HELPER_LIB`** - two metric helpers (`eval_utils`, `spac_edist_align`) that the scoring code
  imports from an absolute path baked into `spacseq_eval_common.EVAL_UTILS_DIR`. The module puts
  that directory on the path itself; it just has to exist.

A checkpoint directory must contain **both** `model.pt` and `pert_map.json`. The perturbation
vocabulary is *not* rebuilt at evaluation time - it is read from `pert_map.json`, so that the
integer code of every perturbation matches the one the weights were trained with.

In [ ]:
import os, sys, json
from pathlib import Path

# code
# The repository root: the directory holding models/ and benchmark_evaluation/. Found by
# walking up from the working directory, so the notebook runs from anywhere inside the repo;
# set CONCERT_CODEBASE to override. Every other root below reads an environment variable too,
# with the paths these results were produced on as the fallback -- see datasets/README.md.
def _repo_root():
    env = os.environ.get("CONCERT_CODEBASE")
    if env:
        return Path(env)
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "models" / "concert_benchmark").is_dir():
            return d
    raise RuntimeError("cannot locate the repository root; set CONCERT_CODEBASE")

def _root(var, default):
    return Path(os.environ.get(var, default))

CODEBASE    = _repo_root()
MODEL_SRC   = CODEBASE / "models" / "concert_benchmark"      # the model this checkpoint is
EVAL_SRC    = CODEBASE / "benchmark_evaluation"              # shared scoring code
SCORING_LIB = _root("CONCERT_SCORING_LIB", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/spac_seq_eval")
# Two metric helpers (eval_utils, spac_edist_align) are imported from an absolute path baked
# into spacseq_eval_common.EVAL_UTILS_DIR. It is not on sys.path here -- the module inserts it
# itself -- but it has to exist, so it is checked below.
HELPER_LIB  = _root("CONCERT_HELPER_LIB", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/CONCERT-main/src/CONCERT-spac")

# data
SPAC_RAW  = _root("CONCERT_DATA_ROOT", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/datasets") / "newdata" / "spac_seq"
PREPARED  = SCORING_LIB / "multi_1-5" / "prepared"           # matched cell sets used at scoring
BENCH_V4  = PREPARED / "benchmark_v4"                         # cross-slide panel + shared source cells

# trained checkpoints (clonality gate + slide guide shift)
OUTPUTS = _root("CONCERT_OUTPUT_ROOT", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/CONCERT/outputs")
CKPT = {
    "seen":   OUTPUTS / "ms_spac1_niche_seen_clonegate_v4"   / "model.pt",
    "unseen": OUTPUTS / "ms_spac1_niche_unseen_clonegate_v4" / "model.pt",
}

# results that came with the paper
SHIPPED = {
    "seen":   SCORING_LIB / "multi_1-5_ho_v4_sampled" / "concert_clonegate" / "seen",
    "unseen": SCORING_LIB / "multi_1-5_ho_v4_sampled" / "concert_clonegate" / "unseen",
}
REPORTED = SCORING_LIB / "v4_aggregate_sampled"               # the reported per-record tables

# where THIS notebook writes
WORKDIR = Path(os.environ.get(
    "CONCERT_NB_WORKDIR",
    "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/notebook_runs/01_spac_seq"))
WORKDIR.mkdir(parents=True, exist_ok=True)

SLIDES = ["1-1", "1-2", "1-3", "1-4", "1-5"]

checks = [
    ("model source",          MODEL_SRC / "concert_spac.py"),
    ("scoring code",          EVAL_SRC / "run_concert_v4_benchmark_eval.py"),
    ("metric helper lib",     HELPER_LIB / "spac_edist_align.py"),
    ("raw section 1-1",       SPAC_RAW / "raw_output_1-1.h5ad"),
    ("prepared cell sets",    PREPARED / "slide_1-1.h5ad"),
    ("cross-slide panel",     BENCH_V4 / "eval_panel.json"),
    ("seen checkpoint",       CKPT["seen"]),
    ("seen vocabulary",       CKPT["seen"].parent / "pert_map.json"),
    ("unseen checkpoint",     CKPT["unseen"]),
    ("unseen vocabulary",     CKPT["unseen"].parent / "pert_map.json"),
    ("shipped seen result",   SHIPPED["seen"] / "patch" / "slide_1-1" / "lfc_scores.json"),
    ("shipped unseen result", SHIPPED["unseen"] / "unseen_pert"),
    ("reported table",        REPORTED / "spac_seen_lfc_dist_per_record.csv"),
]
for label, p in checks:
    print(f"{'OK ' if p.exists() else 'MISSING':>8}  {label:<22}  {p}")
print(f"\nnotebook output dir: {WORKDIR}")

## 2. Import the model and scoring code

**The import order matters.** `run_concert_v4_benchmark_eval.py` puts the live working trees at
the front of `sys.path` when it is imported, and older copies of the same file names exist there.
Importing the shared modules *first*, with the release directories at the front of the path, binds
the released copies into `sys.modules`; the entry point then reuses them. The printed `__file__`
values, and the assertion under them, are the check that the release copies are what got loaded.

In [ ]:
for p in (str(EVAL_SRC), str(MODEL_SRC)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(EVAL_SRC))      # released scoring code wins over the working tree
sys.path.insert(0, str(MODEL_SRC))     # released model source wins over everything

import numpy as np
import pandas as pd
import torch
import eval_spatial_counterfactual as ecf      # binds the released model copy first
import spacseq_eval_common as sec
import spacseq_multi_common as smc
import benchmark_v4_eval_lib as b4
import genefix_wrap
import run_concert_v4_benchmark_eval as bench  # the entry point the SLURM job runs

eu, sa = sec.import_eval_utils()

loaded = {"model source": ecf, "scoring common": sec, "multi-slide common": smc,
          "v4 library": b4, "gene rule": genefix_wrap, "scoring entrypoint": bench}
for k, m in loaded.items():
    print(f"{k:<20}: {m.__file__}")
assert all(Path(m.__file__).resolve().is_relative_to(CODEBASE.resolve()) for m in loaded.values()), \
    "a module was loaded from outside the release"
print(f"{'metric helpers':<20}: {eu.__file__}")
print(f"{'torch':<20}: {torch.__version__}")
print(f"{'device':<20}:", "cuda" if torch.cuda.is_available() else "CPU ONLY - too slow, get a GPU node")
if torch.cuda.is_available():
    print(f"{'gpu':<20}: {torch.cuda.get_device_name(0)}")

### The shared gene rule

The SLURM job runs the entry point through `genefix_wrap.py`, which narrows the scored genes to
the slide's highly variable genes **inside the 2,000-gene CONCERT panel** (the knocked-out gene is
removed by the scoring code itself). Every method in the benchmark is scored over the same genes.
The cell below installs the same rule in this kernel, by wrapping `select_hvgs` exactly as the
wrapper does.

In [ ]:
CONCERT_GENES = set(genefix_wrap.load_concert_genes())
_select_hvgs_orig = getattr(sec, "_select_hvgs_orig", sec.select_hvgs)   # idempotent on re-run
sec._select_hvgs_orig = _select_hvgs_orig

def select_hvgs_genefix(adata_all, n_top, sa_):
    mask = _select_hvgs_orig(adata_all, n_top, sa_)
    names = np.asarray(adata_all.var_names).astype(str)
    return np.asarray(mask, dtype=bool) & np.isin(names, list(CONCERT_GENES))

sec.select_hvgs = select_hvgs_genefix
print(f"gene rule installed: slide HVGs AND {len(CONCERT_GENES)} CONCERT genes, minus the knockout")

## 3. The evaluation configuration

The configuration is built by the entry point's own argument parser, from the same flags
`experiments/01_spac_seq_benchmark/evaluation/eval_spac_seq_benchmark_v4.sh` passes, so the
notebook and the SLURM job cannot drift apart.

Three of them are worth calling out, because getting them wrong fails *silently* rather than
loudly:

* **`fpert_ls_min` / `fpert_ls_max` - the footprint lengthscale band.** The per-perturbation
  lengthscale is `ls_min + (ls_max - ls_min) * sigmoid(ls_head(e_p))`. The band is a plain float
  attribute of the field, not a saved weight, so identical trained weights render a **different
  spatial field** if it is left at the default. It has to be passed at evaluation as well as at
  training.
* **`cf_output poisson_depth`** - the scored prediction is one Poisson draw at each source spot's
  measured library size. The draw is seeded by the perturbation code, so a rerun reproduces the
  shipped numbers exactly.
* **`cf_seen_ref observed`** - the predicted LFC is taken against the measured source spots, the
  count-level reference every competitor receives.

The inducing-grid size and the niche-FiLM head are read out of the checkpoint, and the slide guide
shift of a clonality-gated checkpoint is applied automatically.

`n_boot` / `boot_frac` control the bootstrap that produces the spread on each score: within each
niche the source cells are resampled at 50%, ten times.

In [ ]:
def spac_args(regime, target_slide=None, outdir=None, units=None):
    """Evaluation configuration for one SPAC-seq run, parsed by the entry point itself."""
    argv = ["run_concert_v4_benchmark_eval.py", "--regime", regime,
            "--prepared_dir", str(PREPARED), "--model_file", str(CKPT[regime]),
            "--outdir", str(outdir or (WORKDIR / f"{regime}_eval")),
            "--n_boot", "10", "--min_g_cells", "20",
            "--ls_prior_min", "3", "--ls_prior_max", "18",
            "--fpert_ls_min", "0.15", "--fpert_ls_max", "0.90",
            "--cf_output", "poisson_depth", "--cf_seen_ref", "observed", "--force"]
    if regime == "unseen":
        argv += ["--benchmark_dir", str(BENCH_V4), "--max_src_cells", "5000"]
        if units:
            argv += ["--units", *units]
    else:
        argv += ["--target_slide", target_slide, "--max_src_cells_per_niche", "500"]
    saved = sys.argv
    try:
        sys.argv = argv
        return bench.parse_args()
    finally:
        sys.argv = saved

SLIDE = "1-1"
args = spac_args("seen", target_slide=SLIDE, outdir=WORKDIR / "seen_eval")
print(f"scoring {args.regime} / slide {SLIDE}")
print(f"checkpoint  {args.model_file}")
print(f"output      {args.cf_output}, LFC reference = {args.cf_seen_ref}")
print(f"band        [{args.fpert_ls_min}, {args.fpert_ls_max}] normalized "
      f"= [{args.fpert_ls_min * args.fpert_coord_scale:.0f}, "
      f"{args.fpert_ls_max * args.fpert_coord_scale:.0f}] section units")

## 4. Load the checkpoint

`build_concert_ctx` does four things:

1. reads all five sections with the shared highly-variable-gene mask,
2. maps every cell's perturbation label to its trained integer code using `pert_map.json`,
3. keeps only the target slide (after the vocabulary is built, so the codes stay valid),
4. constructs the model graph and calls `load_model` on it.

What comes back is a plain dict - the evaluation context. The keys used below are:

| key | meaning |
|---|---|
| `model` | the loaded model, in `eval()` mode |
| `N` | cells on the target slide |
| `Y_obs` | raw counts, same rows |
| `gene_names` | the 2000 genes the model predicts |
| `perturb_raw` | each cell's perturbation label |
| `cls` | each cell's class: `real` / `control` / `unassigned` |
| `name2code` | perturbation name -> trained integer code (code 0 is the basal slot) |

This takes a couple of minutes: it reads all five sections.

In [ ]:
%%time
ctx = bench.build_concert_ctx(args, SLIDE)

model = ctx["model"]
print()
print(f"slide {SLIDE}: {ctx['N']:,} cells x {len(ctx['gene_names'])} genes")
print(f"perturbation vocabulary: {ctx['n_batch']} codes (code 0 = basal)")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
import collections
print("cell classes:", dict(collections.Counter(ctx["cls"])))

### The clonality gate

A knockout clone is the set of cells of one perturbation on one slide. Clones with fewer than 100
training cells get weight 0 in the reconstruction loss and in the perturbation targets, so a
perturbation's response is learned from clones large enough to estimate it. Training writes the
per-slide accounting next to the checkpoint.

In [ ]:
gate = json.loads((CKPT["seen"].parent / "clone_gate_summary.json").read_text())
print("minimum clone size:", gate["min_cells"])
pd.DataFrame(gate["slides"]).T[["ko_clones", "ko_clones_kept", "ko_spots", "ko_spots_kept"]]

## 5. What the weights say about each perturbation

Two per-perturbation quantities are read straight out of the checkpoint. Both are produced by a
small head applied to the 256-dimensional perturbation embedding
`e_p = lord_encoder.s_encoder.perturbation.weight[p]`:

* **footprint lengthscale** `ls(p) = ls_min + (ls_max - ls_min) * sigmoid(ls_head(e_p))` -
  how far the perturbation's spatial effect field reaches. A lengthscale that sits **inside** the
  band rather than pinned against an end is what allows it to be described as *learned* rather
  than *bounded*.
* **niche gain** `g_scale(p) = min(softplus(g_scale_head(e_p)), g_cap)` - how strongly the local
  microenvironment is allowed to reshape this perturbation's response.

No inference is needed for this; it is arithmetic on the state dict.

In [ ]:
sd = torch.load(str(args.model_file), map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd:
    sd = sd["model_state_dict"]

emb  = sd["lord_encoder.s_encoder.perturbation.weight"].float()   # (n_perts, 256)
ls_w = sd["fpert_field.ls_head.weight"].float()
ls_b = sd["fpert_field.ls_head.bias"].float()
gs_w = sd["g_scale_head.weight"].float()
gs_b = sd["g_scale_head.bias"].float()

LS_MIN, LS_MAX, G_CAP, COORD_SCALE = args.fpert_ls_min, args.fpert_ls_max, 3.0, args.fpert_coord_scale
ls_norm = (LS_MIN + (LS_MAX - LS_MIN) * torch.sigmoid(emb @ ls_w.T + ls_b)).squeeze(1).numpy()
g_scale = torch.clamp(torch.nn.functional.softplus(emb @ gs_w.T + gs_b), max=G_CAP).squeeze(1).numpy()

code2name = ctx["pert_name"]
tbl = pd.DataFrame({
    "code": np.arange(len(ls_norm)),
    "perturbation": [code2name.get(i, f"code{i}") for i in range(len(ls_norm))],
    "lengthscale_norm": ls_norm,
    "lengthscale_section_units": ls_norm * COORD_SCALE,
    "niche_gain": g_scale,
}).iloc[1:]                       # code 0 is the basal slot, not a perturbation

print(f"band in normalized units: [{LS_MIN}, {LS_MAX}]")
print(f"learned lengthscales    : {tbl.lengthscale_norm.min():.3f} - {tbl.lengthscale_norm.max():.3f}")
print(f"pinned at either end    : {int(((tbl.lengthscale_norm - LS_MIN).abs() < 1e-3).sum() + ((tbl.lengthscale_norm - LS_MAX).abs() < 1e-3).sum())} of {len(tbl)}")
tbl.sort_values("lengthscale_norm").head(10)

## 6. One counterfactual prediction, by hand

Before running the whole protocol, it helps to see a single prediction.

The decoder is

```
log mu  =  f_basal(x, expression)  +  1[has_guide] * mu_action  +  1[is_real_KO] * f_pert(g, x)
```

The source cells are the held-out patch: cells whose own perturbation was masked out of the
model's view of this slide. `map_prep_to_model_rows` bridges the two indexings - the prepared
cell sets are addressed by barcode, the model by row - by matching pixel coordinates.

The scored prediction is built in two steps, exactly as `--cf_output poisson_depth` does it:

```
p_i       =  mu_i / sum(mu_i)                        each source spot's predicted profile
counts_i  ~  Poisson( p_i * library_size_i )         one draw at the spot's measured depth
model LFC =  log mean(counts)  -  log mean(measured counts of the same source spots)
```

This cell is an **illustration**, not the reported protocol: it pools the whole held-out patch
and compares against all measured cells of that perturbation on the slide, where the reported
score is computed per niche. Step 7 runs the real thing.

In [ ]:
%%time
# 1. the held-out patch and which perturbation dominates each niche in it
adata_t   = smc.load_prepared_slide(args.prepared_dir, SLIDE)
role      = smc.load_patch_roles(args.prepared_dir, SLIDE, adata_t)
niche_dom = smc.patch_dominant_perts(adata_t, role, args.min_g_cells, src_role=args.patch_role)
src_idx   = smc.patch_source_indices(adata_t, role, niche_dom, args.max_src_cells_per_niche,
                                     seed=42, src_role=args.patch_role)
kept_idx, model_rows = bench.map_prep_to_model_rows(SLIDE, np.asarray(adata_t.obs_names), src_idx, ctx)
print(f"{len(niche_dom)} evaluable niches, {len(model_rows)} source cells mapped to model rows")

# 2. pick one perturbation, decode the source cells to it, draw counts at each spot's depth
GENE = sorted(set(niche_dom.values()))[0]
code_g = ctx["name2code"][GENE]
print(f"predicting {GENE} (code {code_g}) on {len(model_rows)} source cells")

def _dense(x):
    return np.asarray(x.toarray() if hasattr(x, "toarray") else x, dtype=np.float64)

src_counts = _dense(ctx["Y_obs"][model_rows])
mu_g       = np.asarray(ecf._predict_swap(ctx, model_rows, code_g, args), dtype=np.float64)
p_g        = mu_g / np.maximum(mu_g.sum(1, keepdims=True), 1e-12)
pred_g     = np.random.default_rng(code_g).poisson(p_g * src_counts.sum(1, keepdims=True))
model_lfc  = eu.pseudobulk_lfc(pred_g, src_counts)

# 3. the measured comparison: cells that really carry GENE, against unassigned cells
ko  = np.where(ctx["perturb_raw"] == GENE)[0]
ref = np.where(ctx["cls"] == "unassigned")[0]
obs_lfc = eu.pseudobulk_lfc(ctx["Y_obs"][ko], ctx["Y_obs"][ref])
print(f"measured: {len(ko)} {GENE} cells vs {len(ref)} unassigned cells")

scores = eu.score_lfc_agreement(model_lfc, obs_lfc)
print("\nslide-level illustration (NOT the reported per-niche score):")
for k, v in scores.items():
    print(f"  {k:<16} {v:.4f}")

In [ ]:
# the scoring modules switch matplotlib to a file backend on import; draw inline here
%matplotlib inline
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(obs_lfc, model_lfc, s=5, lw=0, alpha=0.35, color="#2166ac", rasterized=True)
lim = np.nanpercentile(np.abs(np.r_[obs_lfc, model_lfc]), 99.5)
ax.plot([-lim, lim], [-lim, lim], lw=1, color="0.6", zorder=0)
ax.axhline(0, lw=0.6, color="0.85", zorder=0); ax.axvline(0, lw=0.6, color="0.85", zorder=0)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel(f"measured log fold change ({GENE})", fontsize=13)
ax.set_ylabel("predicted log fold change", fontsize=13)
ax.tick_params(labelsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## 7. Score one slide with the shipped harness

`eval_seen_slide` is exactly what `evaluation/eval_spac_seq_benchmark_v4.sh` runs, one slide per
array task. It

1. finds the niches of the held-out patch and the perturbation dominating each,
2. decodes every source cell to every perturbation in the panel and draws counts at depth,
3. scores each (niche, perturbation) record against the measured cells of that perturbation,
4. bootstraps the source cells within each niche for the spread,
5. writes `lfc_scores.json` (one entry per record), `lfc_agg.json` and `per_pert_summary.json`.

A **record** is one (perturbation, slide, niche) triple, read from `lfc_scores.json`. A record
whose LFC RMSE is undefined (no scorable genes) is dropped, as in the reported tables.

**This is the slow step** - it decodes the full panel. Set `RUN_SLIDE_EVAL = False` to skip it
and read the shipped result for the same slide instead. When it runs, the last lines compare the
recomputed records with the shipped ones.

**A recomputed slide matches the shipped one in distribution, record for record it differs.** Each
prediction averages 10 samples from the latent posterior, and the counts are one Poisson draw at
each spot's depth. With 500 source cells per niche, two runs of the same checkpoint give
per-record LFC Spearman values a few hundredths apart, and occasionally about 0.1 apart. The
records, the scored genes and the measured side are identical between runs; the check below
confirms the record set matches and reports the spread.

In [ ]:
def read_seen_records(root, slides):
    rows = []
    for s in slides:
        p = Path(root) / "patch" / f"slide_{s}" / "lfc_scores.json"
        if not p.exists():
            print(f"  slide {s}: no result at {p}")
            continue
        for gene, recs in json.loads(p.read_text())["L2"].items():
            for r in recs:
                rows.append({"slide": s, "perturbation": gene, "niche": r["niche"],
                             "lfc_spearman": r.get("spearman"), "lfc_rmse": r.get("lfc_dist"),
                             "n_scored_genes": r.get("n_scored_genes")})
    df = pd.DataFrame(rows)
    return df[np.isfinite(df.lfc_rmse.astype(float))].reset_index(drop=True)

In [ ]:
%%time
RUN_SLIDE_EVAL = True     # False -> read the shipped result instead of recomputing

if RUN_SLIDE_EVAL:
    bench.eval_seen_slide(args, SLIDE)
    slide_df = read_seen_records(args.outdir, [SLIDE])
else:
    slide_df = read_seen_records(SHIPPED["seen"], [SLIDE])

print(f"slide {SLIDE}: {len(slide_df)} records, "
      f"mean LFC Spearman {slide_df.lfc_spearman.mean():.4f}, "
      f"mean LFC RMSE {slide_df.lfc_rmse.mean():.4f}")
if RUN_SLIDE_EVAL:
    ship = read_seen_records(SHIPPED["seen"], [SLIDE])
    both = slide_df.merge(ship, on=["slide", "perturbation", "niche"], suffixes=("", "_shipped"))
    d_rho = both.lfc_spearman - both.lfc_spearman_shipped
    d_rmse = both.lfc_rmse - both.lfc_rmse_shipped
    print(f"vs shipped: {len(both)}/{len(ship)} records matched; "
          f"slide mean rho {both.lfc_spearman.mean():.4f} here vs {both.lfc_spearman_shipped.mean():.4f} shipped; "
          f"per-record |d rho| median {d_rho.abs().median():.3f}, max {d_rho.abs().max():.3f}; "
          f"|d RMSE| median {d_rmse.abs().median():.3f}")
slide_df.sort_values("lfc_spearman", ascending=False)

## 8. The five slides together - the within-niche number

The manuscript reports the mean over **knockouts**: the records of each knockout are averaged
first, then the knockouts, so a knockout that dominates several niches counts once. The
Supplementary Tables test CONCERT against each method on the matched **records**. The cell below
reads the five per-slide result files and prints both.

By default it reads the **shipped** results, so it reproduces the published number exactly. Point
`RESULT_ROOT` at `args.outdir` instead once you have run step 7 for all five slides
(`for s in SLIDES: bench.eval_seen_slide(spac_args("seen", s, WORKDIR / "seen_eval"), s)`).

In [ ]:
RESULT_ROOT = SHIPPED["seen"]        # or: WORKDIR / "seen_eval"

seen_df = read_seen_records(RESULT_ROOT, SLIDES)

print(f"{len(seen_df)} records ({seen_df.perturbation.nunique()} knockouts) over {seen_df.slide.nunique()} slides")
print(f"\nWITHIN-NICHE (code name: seen)")
print(f"  LFC Spearman : {seen_df.lfc_spearman.mean():.4f}  "
      f"(s.e.m. {seen_df.lfc_spearman.std(ddof=1) / np.sqrt(len(seen_df)):.4f})")
print(f"  LFC RMSE     : {seen_df.lfc_rmse.mean():.4f}  "
      f"(s.e.m. {seen_df.lfc_rmse.std(ddof=1) / np.sqrt(len(seen_df)):.4f})   <- record level")

per_ko = seen_df.groupby("perturbation")[["lfc_spearman", "lfc_rmse"]].mean()
print(f"  LFC Spearman : {per_ko.lfc_spearman.mean():.4f}  "
      f"(s.e.m. {per_ko.lfc_spearman.std(ddof=1) / np.sqrt(len(per_ko)):.4f})")
print(f"  LFC RMSE     : {per_ko.lfc_rmse.mean():.4f}  "
      f"(s.e.m. {per_ko.lfc_rmse.std(ddof=1) / np.sqrt(len(per_ko)):.4f})   <- per knockout (manuscript)")

seen_df.groupby("slide").agg(n_records=("perturbation", "size"),
                             lfc_spearman=("lfc_spearman", "mean"),
                             lfc_rmse=("lfc_rmse", "mean")).round(4)

## 9. The cross-slide setting

In the cross-slide setting a knockout is removed from one **target slide** and its effect has to
be carried over from the donor slides where it was visible in training. The panel in
`benchmark_v4/eval_panel.json` lists 85 such (knockout, target slide) **units**. Every method is
scored on the same 5,000 source cells per target slide (`benchmark_v4/source_cells/`), and the
reference excludes the unassigned cells of the held-out patches.

`eval_unseen_all` walks the panel; `--units Gene@slide` restricts it. The cell below runs **one**
unit end to end from the unseen checkpoint, checks it against the shipped result, and then reads
the shipped results for the reported number.

In [ ]:
panel = json.loads((BENCH_V4 / "eval_panel.json").read_text())
units = pd.DataFrame([{"gene": u["gene"], "target_slide": u["target_slide"],
                       "donor_slides": ",".join(map(str, u.get("donor_slides", []))),
                       "n_ko_target": u.get("n_ko_target")}
                      for u in panel["eval_units"]])
print(f"{len(units)} cross-slide evaluation units")
print(units.target_slide.value_counts().sort_index().to_dict())
units.head(10)

In [ ]:
def read_unseen_records(root):
    rows = []
    for d in sorted((Path(root) / "unseen_pert").glob("*__slide_*")):
        p = d / "lfc_scores.json"
        if not p.exists():
            continue
        gene, slide = d.name.split("__slide_")
        # lfc_scores.json is {"L2": {gene: [record]}}; a cross-slide unit has exactly one record
        # (niche = -1, the whole slide).
        rec = json.loads(p.read_text())["L2"][gene][0]
        rows.append({"gene": gene, "slide": slide,
                     "n_src_cells": rec.get("n_src"), "n_measured_ko": rec.get("n_real_g"),
                     "lfc_spearman": rec.get("spearman"), "lfc_rmse": rec.get("lfc_dist")})
    return pd.DataFrame(rows).dropna(subset=["lfc_spearman"])

In [ ]:
%%time
RUN_ONE_UNSEEN_UNIT = True    # runs a single unit end-to-end from the unseen checkpoint

if RUN_ONE_UNSEEN_UNIT:
    u0 = panel["eval_units"][0]
    unit = f"{u0['gene']}@{u0['target_slide']}"
    u_args = spac_args("unseen", outdir=WORKDIR / "unseen_eval_demo", units=[unit])
    bench.eval_unseen_all(u_args)
    demo = read_unseen_records(u_args.outdir)
    ship = read_unseen_records(SHIPPED["unseen"])
    cmp_ = demo.merge(ship, on=["gene", "slide"], suffixes=("", "_shipped"))
    display(cmp_[["gene", "slide", "lfc_spearman", "lfc_spearman_shipped", "lfc_rmse", "lfc_rmse_shipped"]])

In [ ]:
unseen_df = read_unseen_records(SHIPPED["unseen"])

print("CROSS-SLIDE (code name: unseen)")
print(f"  {len(unseen_df)} units")
print(f"  LFC Spearman : {unseen_df.lfc_spearman.mean():.4f}")
print(f"  LFC RMSE     : {unseen_df.lfc_rmse.mean():.4f}")
unseen_df.sort_values("lfc_spearman", ascending=False)

## 10. CONCERT next to the published methods and the reference strategies

Every method is rescored under **one** rule into a per-record CSV, so the comparison does not
depend on each method's own reporting conventions. The reported comparison has three parts, all
in `v4_aggregate_sampled/`:

* `spac_<setting>_lfc_dist_per_record.csv` - CONCERT and the six published methods,
* `spac_<setting>_naive_baselines_our_rule.csv` - the four reference strategies (Mean target is
  the mean of 50 randomly drawn knockout cells per unit),
* `spac_<setting>_state_our_rule.csv` - STATE.

**The comparison is paired, one method at a time.** The methods cover very different subsets
of the records - GEARS 80 of the 129 within-niche records, CelComen 15, CellAgentChat 11 - so
intersecting *all* of them at once leaves nothing to compare. For each method separately, the
reported comparison takes the records that method and CONCERT both cover and compares the two on
exactly those. The `n_pairs` column of the shipped statistics table is that count, and the cells
below reproduce it.

In [ ]:
CONCERT_KEY = {"seen": "concert_nichefix_seen", "unseen": "concert_v4_unseen"}

def load_all_methods(setting):
    parts = [REPORTED / f"spac_{setting}_{n}.csv"
             for n in ("lfc_dist_per_record", "naive_baselines_our_rule", "state_our_rule")]
    df = pd.concat([pd.read_csv(p, dtype={"slide": str, "niche_key": str}) for p in parts], ignore_index=True)
    df["record"] = df.gene.astype(str) + "|" + df.slide + "|" + df.niche_key
    return df

def paired_comparison(setting):
    df = load_all_methods(setting)
    con = df[df.method == CONCERT_KEY[setting]].set_index("record")
    rows = []
    for m, sub in df[df.method != CONCERT_KEY[setting]].groupby("method"):
        sub = sub.set_index("record")
        shared = con.index.intersection(sub.index)
        if len(shared) == 0:
            continue
        rows.append({
            "method": sub.method_label.iloc[0],
            "n_pairs": len(shared),
            "concert_rho": con.loc[shared, "lfc_spearman"].mean(),
            "method_rho": sub.loc[shared, "lfc_spearman"].mean(),
            "concert_rmse": con.loc[shared, "lfc_dist"].mean(),
            "method_rmse": sub.loc[shared, "lfc_dist"].mean(),
        })
    out = pd.DataFrame(rows)
    out["rho_advantage"] = out.concert_rho - out.method_rho
    out["rmse_advantage"] = out.method_rmse - out.concert_rmse   # positive = CONCERT lower
    return out.sort_values("n_pairs", ascending=False).reset_index(drop=True), len(con)

seen_cmp, n_concert = paired_comparison("seen")
print(f"WITHIN-NICHE: CONCERT covers {n_concert} records; paired against each method:")
display(seen_cmp.round(4))

unseen_cmp, n_concert_u = paired_comparison("unseen")
print(f"\nCROSS-SLIDE: CONCERT covers {n_concert_u} records; paired against each method:")
display(unseen_cmp.round(4))

The shipped statistics tables carry the same `n_pairs` per method, with the paired Wilcoxon test
and its Benjamini-Hochberg adjusted p-value. Matching the counts is the check that the pairing
above is the one the paper used.

In [ ]:
for setting, cmp_df in (("seen", seen_cmp), ("unseen", unseen_cmp)):
    shipped_stats = pd.read_csv(REPORTED / f"spacseq_{setting}_concert_vs_competitor_lfc_rho_stats_summary.csv")
    check = cmp_df[["method", "n_pairs"]].merge(
        shipped_stats[["method", "n_pairs", "p_adj"]].rename(columns={"n_pairs": "n_pairs_shipped"}),
        on="method", how="left")
    check["matches"] = check.n_pairs == check.n_pairs_shipped
    print(f"{setting}: all counts match: {bool(check.matches.all())}")
    display(check)

In [ ]:
# the scoring modules switch matplotlib to a file backend on import; draw inline here
%matplotlib inline
fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
for ax, (cmp_df, name) in zip(axes, [(seen_cmp, "within-niche"), (unseen_cmp, "cross-slide")]):
    d = cmp_df.sort_values("method_rho")
    y = np.arange(len(d))
    ax.barh(y - 0.2, d.concert_rho, 0.4, color="#b2182b", label="CONCERT")
    ax.barh(y + 0.2, d.method_rho, 0.4, color="#9ab8d4", label="other method")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{c}  (n={n})" for c, n in zip(d.method, d.n_pairs)])
    ax.axvline(0, lw=0.8, color="0.4")
    ax.set_xlabel(f"LFC Spearman on paired records ({name})", fontsize=13)
    ax.tick_params(labelsize=10)
    ax.legend(fontsize=10)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## What to watch out for

* **The lengthscale band is not in the weights.** `--fpert_ls_min` / `--fpert_ls_max` must be
  passed at evaluation exactly as at training. Leave them out and the same weights render a
  different spatial field - the run completes and produces plausible, wrong numbers.
* **The output transform and the reference go together.** The reported numbers use the Poisson
  draw at depth *and* the measured source spots as the LFC reference, for CONCERT and for every
  other method. Scoring the mean output against measured counts, or a draw against a model-made
  reference, gives a different number that is comparable with nothing in the tables.
* **The vocabulary is not rebuilt.** Evaluation reads `pert_map.json` from beside the checkpoint.
  A checkpoint without it cannot be scored; do not regenerate it from the data.
* **Compare one method at a time, on the records they share.** The methods cover very different
  subsets, and two of them cover about a tenth of the records - an all-method intersection is
  empty here. See step 10. (The Perturb-map benchmark is the other way round: there the methods
  overlap enough that the intersection is the rule.)
* **`seen` / `unseen` are directory names, not manuscript terms.** They are *within-niche
  prediction* and *cross-slide prediction* in text.
* **E-distance is retired.** It still appears in the JSON files. The reported metrics are LFC RMSE
  and LFC Spearman.
* **Storage.** `/n/holylfs06` is close to its group quota, and writes there can fail *and
  truncate the target to zero*. This notebook writes to `/n/holylabs` and only reads from
  holylfs06.

## Where things are

| | |
|---|---|
| model source | `models/concert_benchmark/` |
| scoring code | `benchmark_evaluation/` (`run_concert_v4_benchmark_eval.py`, `genefix_wrap.py`) |
| experiment scripts | `experiments/01_spac_seq_benchmark/` (train, `evaluation/eval_spac_seq_benchmark_v4.sh`, `evaluation/aggregate_spac_seq_benchmark_v4.sh`) |
| reported tables | `<SCORING_LIB>/v4_aggregate_sampled/` |